# 2. Data cleaning and preprocessing - Brazil Dataset

In Brazilian public health data, occurrence of missing or inconsistent data is usual and it mostly occurs due to incorrect filling of handwritten forms. In order to deal with this problem, the proposed method uses two specific approaches: for non-categorical features, with continuous numerical values, the mean value for this feature in the dataset is calculated, and the feature is filled with this value; whereas features with categorical values (discrete values) are filled using the most frequent value for this feature in the dataset. Also, we exclude the ignored data.

## CONTENTS

* [Imports](#Imports)  
* [Load Datasets](#load)
* [Data Split](#slice)
* [Parse Date](#parse)
* [Inconsistencies and Ignored](#remove)
* [Missing Values](#missing)

<a id='Imports'></a>
### Imports

In [2]:
import pandas as pd
import os
import zipfile
import os,glob
import numpy as np
from IPython.display import display
pd.options.display.max_columns = None
pd.options.display.max_rows = None

<a id='load'></a>
### Load datasets

Reading merged dataset filtering some columns of interest

In [3]:
%cd /home/narruda/Merge_Neonatal
df_merge = {}
for filename in glob.glob('*.csv'):
    df_merge[filename[:-4]] = pd.read_csv(filename, sep=",", encoding = "latin-1",low_memory=False, 
                                          usecols=['LOCNASC','DTNASCMAE','RACACORMAE','QTDGESTANT',
                                                   'CODMUNNASC','QTDPARTNOR','QTDPARTCES','SEMAGESTAC_x',
                                                   'IDADEMAE_x','TPMETESTIM','CONSPRENAT','MESPRENAT',
                                                   'ESTCIVMAE','TPAPRESENT','STTRABPART','STCESPARTO',
                                                   'ESCMAE_x','TPROBSON','ESCMAE2010_x','TPNASCASSI',
                                                   'QTDFILVIVO_x','ESCMAEAGR1_x','TPFUNCRESP','PARIDADE',
                                                   'QTDFILMORT_x','KOTELCHUCK','TIPOBITO','DTOBITO','DTNASC_y',
                                                   'CODMUNRES_x','NATURAL','IDADE','SEXO_y','CODBAIRES_y',
                                                   'GESTACAO_x','CODMUNRES_y','OBITOPARTO','PESO_y','ASSISTMED',
                                                   'GRAVIDEZ_x','EXAME','CIRURGIA','NECROPSIA','CAUSABAS',
                                                   'PARTO_x','LINHAA','LINHAB','LINHAC','LINHAD','LINHAII',
                                                   'CONSULTAS','UFINFORM_y','CAUSABAS_O','CAUSAMAT','ESCMAE2010_y',
                                                   'DTNASC_x','CODMUNCART_y','CODCART_y','SEMAGESTAC_y','RACACOR_x',
                                                   'SEXO_x','ESCMAEAGR1_y','ATESTADO','APGAR1','APGAR5','PESO_x',
                                                   'CODANOMAL','IDANOMAL','CODBAINASC','CODBAIRES_x','UFINFORM_x'])                                          

/home/narruda/Merge_Neonatal


In [4]:
for keys in df_merge.keys():
    print(keys,df_merge[keys].shape)

PI (890168, 71)
RJ (3858678, 71)
PE (2489197, 71)
MG (4576468, 71)
MT (862971, 71)
GO (1562105, 71)
TO (430395, 71)
BA (3743909, 71)
AM (1264794, 71)
RN (845472, 71)
DF (765361, 71)
PA (2408488, 71)
AL (973204, 71)
RR (171842, 71)
PR (2680641, 71)
AP (252746, 71)
SE (610847, 71)
SP (6698148, 71)
SC (1492715, 71)
RS (2468241, 71)
MA (2034942, 71)
PB (1022229, 71)
AC (281202, 71)
CE (2282610, 71)
ES (913759, 71)
MS (699715, 71)
RO (463920, 71)


<font size="3">Create target variable identifying death before 28th of life as 1 and non-death as 0</font>

In [5]:
for key, values in df_merge.items():
    df_merge[key]['morte_menor_28d'] = np.where(df_merge[key]['DTOBITO'].isnull(),0,1)
    print(key, df_merge[key][df_merge[key]['morte_menor_28d']==1].shape, df_merge[key][df_merge[key]['morte_menor_28d']==0].shape)

PI (4060, 72) (886108, 72)
RJ (14833, 72) (3843845, 72)
PE (11665, 72) (2477532, 72)
MG (15003, 72) (4561465, 72)
MT (4189, 72) (858782, 72)
GO (4267, 72) (1557838, 72)
TO (1494, 72) (428901, 72)
BA (18104, 72) (3725805, 72)
AM (6695, 72) (1258099, 72)
RN (2294, 72) (843178, 72)
DF (3091, 72) (762270, 72)
PA (12031, 72) (2396457, 72)
AL (2570, 72) (970634, 72)
RR (698, 72) (171144, 72)
PR (12862, 72) (2667779, 72)
AP (1518, 72) (251228, 72)
SE (3951, 72) (606896, 72)
SP (37153, 72) (6660995, 72)
SC (4872, 72) (1487843, 72)
RS (10580, 72) (2457661, 72)
MA (7858, 72) (2027084, 72)
PB (2828, 72) (1019401, 72)
AC (567, 72) (280635, 72)
CE (7657, 72) (2274953, 72)
ES (2125, 72) (911634, 72)
MS (3960, 72) (695755, 72)
RO (1543, 72) (462377, 72)


<font size="3">Create an id variable combining index and state</font>

In [6]:
for key, values in df_merge.items():
    df_merge[key]['id'] = df_merge[key].index
    df_merge[key]['UF_id'] = key[:2]
    df_merge[key]['ID'] = df_merge[key]['id'].map(str) + df_merge[key]['UF_id']

<a id='slice'></a>
### Slice 10% of dataset

In [7]:
df_nasc = {}
nasc_size = {}
df_obito = {}
obito_size = {}
df_obito_sample_10 = {}
df_nasc_sample_10 = {}
    
for key, values in df_merge.items():
    
    df_nasc[key] = df_merge[key][df_merge[key]["morte_menor_28d"] == 0]
    nasc_size[key] = int(df_nasc[key].shape[0] * 0.10)
    
    df_obito[key] = df_merge[key][df_merge[key]["morte_menor_28d"] == 1]
    obito_size[key] = int(df_obito[key].shape[0] * 0.10)
    
    df_obito_sample_10[key] = df_obito[key].sample(obito_size[key], random_state = 1)
    df_nasc_sample_10[key] = df_nasc[key].sample(nasc_size[key], random_state = 1)

    print(key,nasc_size[key],obito_size[key])

PI 88610 406
RJ 384384 1483
PE 247753 1166
MG 456146 1500
MT 85878 418
GO 155783 426
TO 42890 149
BA 372580 1810
AM 125809 669
RN 84317 229
DF 76227 309
PA 239645 1203
AL 97063 257
RR 17114 69
PR 266777 1286
AP 25122 151
SE 60689 395
SP 666099 3715
SC 148784 487
RS 245766 1058
MA 202708 785
PB 101940 282
AC 28063 56
CE 227495 765
ES 91163 212
MS 69575 396
RO 46237 154


In [8]:
df_10 = {}
for (key, values), (key1, values1) in zip(df_obito_sample_10.items(),df_nasc_sample_10.items()):
    df_10[key] = pd.concat([df_obito_sample_10[key],df_nasc_sample_10[key1]])
    print(key, df_10[key].shape)

PI (89016, 75)
RJ (385867, 75)
PE (248919, 75)
MG (457646, 75)
MT (86296, 75)
GO (156209, 75)
TO (43039, 75)
BA (374390, 75)
AM (126478, 75)
RN (84546, 75)
DF (76536, 75)
PA (240848, 75)
AL (97320, 75)
RR (17183, 75)
PR (268063, 75)
AP (25273, 75)
SE (61084, 75)
SP (669814, 75)
SC (149271, 75)
RS (246824, 75)
MA (203493, 75)
PB (102222, 75)
AC (28119, 75)
CE (228260, 75)
ES (91375, 75)
MS (69971, 75)
RO (46391, 75)


In [9]:
df_Brasil_sample_10 = pd.DataFrame()
for key, values in df_10.items():
    df_Brasil_sample_10 = df_Brasil_sample_10.append(df_10[key])

/home/narruda/.local/lib/python3.6/site-packages/pandas/core/frame.py:7123: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  sort=sort,


In [10]:
df_Brasil_sample_10.shape

(4674453, 75)

In [ ]:
#df_Brasil_sample_10.to_csv('df_brasil_sample_10', index = False)

In [11]:
df_90 = {}
for key,values in df_merge.items():
    df_90[key] = pd.concat([df_merge[key], df_nasc_sample_10[key], df_obito_sample_10[key]])
    
    print(key, df_90[key].shape)

PI (979184, 75)
RJ (4244545, 75)
PE (2738116, 75)
MG (5034114, 75)
MT (949267, 75)
GO (1718314, 75)
TO (473434, 75)
BA (4118299, 75)
AM (1391272, 75)
RN (930018, 75)
DF (841897, 75)
PA (2649336, 75)
AL (1070524, 75)
RR (189025, 75)
PR (2948704, 75)
AP (278019, 75)
SE (671931, 75)
SP (7367962, 75)
SC (1641986, 75)
RS (2715065, 75)
MA (2238435, 75)
PB (1124451, 75)
AC (309321, 75)
CE (2510870, 75)
ES (1005134, 75)
MS (769686, 75)
RO (510311, 75)


In [12]:
for key,values in df_90.items():
    df_90[key] = df_90[key].drop_duplicates(keep=False)
    print(key, df_90[key].shape)

PI (801152, 75)
RJ (3472811, 75)
PE (2240278, 75)
MG (4118822, 75)
MT (776675, 75)
GO (1405896, 75)
TO (387356, 75)
BA (3369519, 75)
AM (1138316, 75)
RN (760926, 75)
DF (688825, 75)
PA (2167640, 75)
AL (875884, 75)
RR (154659, 75)
PR (2412578, 75)
AP (227473, 75)
SE (549763, 75)
SP (6028334, 75)
SC (1343444, 75)
RS (2221417, 75)
MA (1831449, 75)
PB (920007, 75)
AC (253083, 75)
CE (2054350, 75)
ES (822384, 75)
MS (629744, 75)
RO (417529, 75)


In [13]:
for key, values in df_90.items():
    
    assert pd.merge(df_90[key], df_10[key], on="id", how="inner").shape[0] == 0

<font size="3">Rename columns</font>

In [14]:
for key, values in df_90.items():
    df_90[key] = df_90[key].rename(columns={'LOCNASC':'n_tp_ocorrencia',
                                            'CODMUNNASC':'n_co_municipio_ibge_ocorrencia',
                                            'IDADEMAE_x':'n_nu_idade',
                                            'ESTCIVMAE':'n_tp_estado_civil',
                                            'ESCMAE_x':'n_tp_escolaridade',
                                            'QTDFILVIVO_x':'n_qt_nascidos_vivos',
                                            'QTDFILMORT_x':'n_qt_nascidos_mortos',
                                            'CODMUNRES_x':'n_co_municipio_ibge_residencia',
                                            'GESTACAO_x':'n_tp_gestacao',
                                            'GRAVIDEZ_x':'n_tp_gravidez',
                                            'PARTO_x':'n_tp_parto',
                                            'CONSULTAS':'n_tp_prenatal',
                                            'DTNASC_x':'n_dt_nascimento',
                                            'SEXO_x':'n_sg_sexo',
                                            'APGAR1':'n_nu_apgar1',
                                            'APGAR5':'n_nu_apgar5',
                                            'RACACOR_x':'n_tp_raca_cor',
                                            'PESO_x':'n_nu_peso',
                                            'CODANOMAL':'n_co_cid',
                                            'IDANOMAL':'n_st_malformacao',
                                            'CODBAINASC':'n_co_bairro_ocorrencia',
                                            'CODBAIRES_x':'n_co_bairro_residencia',
                                            'UFINFORM_x':'n_nu_uf_inform',
                                            'DTNASCMAE':'n_dt_nascimento_mae',
                                            'RACACORMAE':'n_tp_raca_cor_mae',
                                            'QTDGESTANT':'n_qt_gestacao_anterior',
                                            'QTDPARTNOR':'n_qt_parto_normal',
                                            'QTDPARTCES':'n_qt_parto_cesaria',
                                            'DTULTMENST':'n_dt_ultima_menstruacao',
                                            'SEMAGESTAC_x':'n_nu_semana_gestacao',
                                            'TPMETESTIM':'n_tp_metodo_estimar',
                                            'CONSPRENAT':'n_nu_consulta_prenatal',
                                            'MESPRENAT':'n_nu_mes_gestacao_prenatal',
                                            'TPAPRESENT':'n_tp_apresentacao',
                                            'STTRABPART':'n_st_trabalho_parto',
                                            'STCESPARTO':'n_st_cesarea_parto',
                                            'TPROBSON':'n_tp_grupo_robson',
                                            'ESCMAE2010_x':'n_tp_escolaridade_2010',
                                            'TPNASCASSI':'n_tp_nascimento_assistido',
                                            'ESCMAEAGR1_x':'n_tp_escolaridade_agregado1',
                                            'TPFUNCRESP':'n_tp_funcao_responsavel',
                                            'PARIDADE':'n_paridade',
                                            'KOTELCHUCK':'n_nu_kotelchuck',
                                            'TIPOBITO':'o_tp_obito',
                                            'DTOBITO':'o_dt_obito',
                                            'NATURAL':'o_co_naturalidade',
                                            'DTNASC_y':'o_dt_nascimento',
                                            'IDADE':'o_nu_idade',
                                            'SEXO_y':'o_sg_sexo',
                                            'CODBAIRES_y':'o_co_bairro_residencia',
                                            'CODMUNRES_y':'o_co_municipio_ibge_residencia',
                                            'OBITOPARTO':'o_tp_morte_parto',
                                            'PESO_y':'o_nu_peso',
                                            'ASSISTMED':'o_st_assistencia_medica',
                                            'EXAME':'o_st_exame_complementar',
                                            'CIRURGIA':'o_st_cirurgia',
                                            'NECROPSIA':'o_st_necropsia',
                                            'CAUSABAS':'o_co_cid_causa_basica',
                                            'LINHAA':'o_ds_causa_morte_a',
                                            'LINHAB':'o_ds_causa_morte_b',
                                            'LINHAC':'o_ds_causa_morte_c',
                                            'LINHAD':'o_ds_causa_morte_d',
                                            'LINHAII':'o_ds_causa_morte_2',
                                            'UFINFORM_y':'o_uf_nu_inform',
                                            'CAUSABAS_O':'o_causabasic_o',
                                            'CAUSAMAT':'o_nu_causamat',
                                            'ESCMAE2010_y':'o_tp_escolaridade_2010_mae',
                                            'CODMUNCART_y':'o_co_municipio_ibge_cartorio',
                                            'CODCART_y':'o_codcart',
                                            'ESCMAEAGR1_y':'o_tp_escolaridademae_agregado1',
                                            'SEMAGESTAC_y':'o_nu_semana_gestacao',
                                            'ATESTADO':'o_co_cid_causa_morte'})

<a id='parse'></a>
### Parse date fields

In [15]:
for key, values in df_90.items():
    df_90[key]['o_dt_obito'] = df_90[key]['o_dt_obito'].astype('Int64')
    df_90[key]['o_dt_nascimento'] = df_90[key]['o_dt_nascimento'].astype('Int64')
    df_90[key]['n_dt_nascimento'] = df_90[key]['n_dt_nascimento'].astype('Int64')
    print(key, df_90[key].shape, df_90[key]['o_dt_obito'].dtype, df_90[key]['o_dt_nascimento'].dtype, df_90[key]['o_dt_nascimento'].dtype)

PI (801152, 75) Int64 Int64 Int64
RJ (3472811, 75) Int64 Int64 Int64
PE (2240278, 75) Int64 Int64 Int64
MG (4118822, 75) Int64 Int64 Int64
MT (776675, 75) Int64 Int64 Int64
GO (1405896, 75) Int64 Int64 Int64
TO (387356, 75) Int64 Int64 Int64
BA (3369519, 75) Int64 Int64 Int64
AM (1138316, 75) Int64 Int64 Int64
RN (760926, 75) Int64 Int64 Int64
DF (688825, 75) Int64 Int64 Int64
PA (2167640, 75) Int64 Int64 Int64
AL (875884, 75) Int64 Int64 Int64
RR (154659, 75) Int64 Int64 Int64
PR (2412578, 75) Int64 Int64 Int64
AP (227473, 75) Int64 Int64 Int64
SE (549763, 75) Int64 Int64 Int64
SP (6028334, 75) Int64 Int64 Int64
SC (1343444, 75) Int64 Int64 Int64
RS (2221417, 75) Int64 Int64 Int64
MA (1831449, 75) Int64 Int64 Int64
PB (920007, 75) Int64 Int64 Int64
AC (253083, 75) Int64 Int64 Int64
CE (2054350, 75) Int64 Int64 Int64
ES (822384, 75) Int64 Int64 Int64
MS (629744, 75) Int64 Int64 Int64
RO (417529, 75) Int64 Int64 Int64


In [16]:
for key, values in df_90.items():
    df_90[key]['n_dt_nascimento'] = df_90[key]['n_dt_nascimento'].apply(lambda x: '{0:0>8}'.format(x))
    df_90[key]['o_dt_obito'] = df_90[key]['o_dt_obito'].apply(lambda x: np.NaN if type(x) == float else '{0:0>8}'.format(x))
    df_90[key]['o_dt_nascimento'] = df_90[key]['o_dt_nascimento'].apply(lambda x: np.NaN if type(x) == float else '{0:0>8}'.format(x))

In [17]:
for key, values in df_90.items():
    df_90[key]["date_birth"] = pd.to_datetime(
    df_90[key]["n_dt_nascimento"], format="%d%m%Y", errors="coerce"
    )
    df_90[key]["date_death"] = pd.to_datetime(
    df_90[key]["o_dt_obito"], format="%d%m%Y", errors="coerce"
    )
    df_90[key]["o_date_birth"] = pd.to_datetime(
    df_90[key]["o_dt_nascimento"], format="%d%m%Y", errors="coerce"
    )

In [18]:
# intended to check if the date parsing is correct
for key, values in df_90.items():
    check = df_90[key][
        ((df_90[key]["date_death"] - df_90[key]["date_birth"]).dt.days <= 28)
        & ((df_90[key]["date_death"] - df_90[key]["date_birth"]).dt.days >= 0)
        & df_90[key]["morte_menor_28d"]
        == 1][["date_death", "date_birth"]]
    print(key, check.shape)

PI (3606, 2)
RJ (13225, 2)
PE (10381, 2)
MG (13347, 2)
MT (3744, 2)
GO (3811, 2)
TO (1332, 2)
BA (16083, 2)
AM (5944, 2)
RN (2048, 2)
DF (2747, 2)
PA (10627, 2)
AL (2287, 2)
RR (625, 2)
PR (11496, 2)
AP (1356, 2)
SE (3533, 2)
SP (33262, 2)
SC (4338, 2)
RS (9443, 2)
MA (6975, 2)
PB (2532, 2)
AC (509, 2)
CE (6819, 2)
ES (1893, 2)
MS (3530, 2)
RO (1376, 2)


#### Keep only records without inconsistencies between SIM and SINASC birth dates

In [19]:
for key, values in df_90.items():
    df = df_90[key]
    df = df.drop(df[(df.o_date_birth != df.date_birth) & df.morte_menor_28d ==1].index)
    df_90[key] = df
    
    print(key, df.shape,df[df['morte_menor_28d']==1].shape)

PI (801000, 78) (3502, 78)
RJ (3472406, 78) (12945, 78)
PE (2239899, 78) (10120, 78)
MG (4118343, 78) (13024, 78)
MT (776535, 78) (3631, 78)
GO (1405770, 78) (3715, 78)
TO (387310, 78) (1299, 78)
BA (3368876, 78) (15651, 78)
AM (1138051, 78) (5761, 78)
RN (760856, 78) (1995, 78)
DF (688726, 78) (2683, 78)
PA (2167053, 78) (10241, 78)
AL (875803, 78) (2232, 78)
RR (154639, 78) (609, 78)
PR (2412355, 78) (11353, 78)
AP (227422, 78) (1316, 78)
SE (549682, 78) (3475, 78)
SP (6027652, 78) (32756, 78)
SC (1343316, 78) (4257, 78)
RS (2221184, 78) (9289, 78)
MA (1831142, 78) (6766, 78)
PB (919920, 78) (2459, 78)
AC (253066, 78) (494, 78)
CE (2054105, 78) (6647, 78)
ES (822322, 78) (1851, 78)
MS (629641, 78) (3461, 78)
RO (417475, 78) (1335, 78)


In [20]:
for key, values in df_90.items():
    df_90[key]["day_birth"] = df_90[key]["date_birth"].dt.day
    df_90[key]["month_birth"] = df_90[key]["date_birth"].dt.month
    df_90[key]["year_birth"] = df_90[key]["date_birth"].dt.year

    df_90[key]["day_death"] = df_90[key]["date_death"].dt.day
    df_90[key]["month_death"] = df_90[key]["date_death"].dt.month
    df_90[key]["year_death"] = df_90[key]["date_death"].dt.year
    
    print(key, df_90[key][df_90[key]['morte_menor_28d']==0].shape,df_90[key][df_90[key]['morte_menor_28d']==1].shape)

PI (797498, 84) (3502, 84)
RJ (3459461, 84) (12945, 84)
PE (2229779, 84) (10120, 84)
MG (4105319, 84) (13024, 84)
MT (772904, 84) (3631, 84)
GO (1402055, 84) (3715, 84)
TO (386011, 84) (1299, 84)
BA (3353225, 84) (15651, 84)
AM (1132290, 84) (5761, 84)
RN (758861, 84) (1995, 84)
DF (686043, 84) (2683, 84)
PA (2156812, 84) (10241, 84)
AL (873571, 84) (2232, 84)
RR (154030, 84) (609, 84)
PR (2401002, 84) (11353, 84)
AP (226106, 84) (1316, 84)
SE (546207, 84) (3475, 84)
SP (5994896, 84) (32756, 84)
SC (1339059, 84) (4257, 84)
RS (2211895, 84) (9289, 84)
MA (1824376, 84) (6766, 84)
PB (917461, 84) (2459, 84)
AC (252572, 84) (494, 84)
CE (2047458, 84) (6647, 84)
ES (820471, 84) (1851, 84)
MS (626180, 84) (3461, 84)
RO (416140, 84) (1335, 84)


### Check point

<font size="3"> Save dataset with 90% slice sample with ignored and missing</font>

In [ ]:
for key,values in df_90.items():
    df_90[key].to_csv(key+"_sample_90.csv", index=False)
    print(key)

In [6]:
%cd /home/narruda/Merge_Neonatal/Merge_sample_90/Dataset_90_with_death_variables

df_merge = {}
for filename in glob.glob('*.csv'):
    df_merge[filename[:-4]] = pd.read_csv(filename, sep=",", encoding = "latin-1",low_memory=False) 

/home/narruda/Merge_Neonatal/Merge_sample_90/Dataset_90_with_death_variables


<a id='remove'></a>
### Remove inconsistencies / Ignored data

In [8]:
df_90_with_ignored = {}

for key, values in df_merge.items():
    df_90_with_ignored[key] = df_merge[key].copy()
    df = df_90_with_ignored[key]
        
    df = df.drop(df[(df['n_tp_ocorrencia'] == 9) | (df['n_tp_estado_civil'] == 9) |
                    (df['n_tp_escolaridade'] == 9) | (df['n_tp_gestacao'] == 9) |
                    (df['n_tp_gravidez'] == 9)|(df['n_tp_parto'] == 9)| 
                    (df['n_tp_prenatal'] == 9) | (df['n_tp_raca_cor'] == 9) | 
                    (df['n_st_malformacao'] == 9) | (df['n_tp_raca_cor_mae'] == 9) |
                    (df['n_tp_metodo_estimar'] == 9) | (df['n_tp_apresentacao'] == 9) | 
                    (df['n_st_trabalho_parto'] == 9) | (df['n_st_cesarea_parto'] == 9)|
                    (df['n_tp_nascimento_assistido'] == 9)].index)
                        
    df_90_with_ignored[key] = df
        
    print(key, df[df['morte_menor_28d']==1].shape, df[df['morte_menor_28d']==0].shape)

CE_sample_90 (5981, 84) (1294654, 84)
MA_sample_90 (6163, 84) (1216394, 84)
MT_sample_90 (3142, 84) (510565, 84)
MG_sample_90 (11162, 84) (2565285, 84)
PE_sample_90 (9441, 84) (1459707, 84)
PB_sample_90 (2126, 84) (583199, 84)
ES_sample_90 (1563, 84) (488305, 84)
PA_sample_90 (9397, 84) (1435562, 84)
RJ_sample_90 (10649, 84) (2102981, 84)
RR_sample_90 (558, 84) (105572, 84)
PR_sample_90 (10196, 84) (1518527, 84)
SP_sample_90 (29642, 84) (5603820, 84)
RO_sample_90 (1190, 84) (268614, 84)
BA_sample_90 (13257, 84) (2043223, 84)
SC_sample_90 (3938, 84) (901137, 84)
DF_sample_90 (2045, 84) (402882, 84)
AC_sample_90 (378, 84) (153525, 84)
RS_sample_90 (8411, 84) (1399749, 84)
TO_sample_90 (1132, 84) (247399, 84)
RN_sample_90 (1610, 84) (459667, 84)
AP_sample_90 (1109, 84) (140032, 84)
GO_sample_90 (2517, 84) (845654, 84)
MS_sample_90 (3278, 84) (428828, 84)
PI_sample_90 (3214, 84) (513592, 84)
AM_sample_90 (4706, 84) (703043, 84)
AL_sample_90 (2039, 84) (547216, 84)
SE_sample_90 (3306, 84) (

In [9]:
for key, values in df_90_with_ignored.items():
    df = df_90_with_ignored[key]
    df = df.drop(df[(df['n_nu_idade'] == 99)|(df['n_qt_nascidos_vivos'] == 99)|
                    (df['n_qt_nascidos_mortos'] == 99)|(df['n_nu_apgar1'] == 99)|
                    (df['n_nu_apgar5'] == 99)|(df['n_qt_gestacao_anterior'] == 99)|
                    (df['n_qt_parto_normal'] == 99)|(df['n_qt_parto_cesaria'] == 99)].index)
          
    df_90_with_ignored[key] = df
    print(key, df[df['morte_menor_28d']==1].shape, df[df['morte_menor_28d']==0].shape)

CE_sample_90 (5971, 84) (1292959, 84)
MA_sample_90 (6154, 84) (1214487, 84)
MT_sample_90 (3140, 84) (510198, 84)
MG_sample_90 (11119, 84) (2561412, 84)
PE_sample_90 (9434, 84) (1459003, 84)
PB_sample_90 (2121, 84) (582887, 84)
ES_sample_90 (1558, 84) (487978, 84)
PA_sample_90 (9280, 84) (1419842, 84)
RJ_sample_90 (10556, 84) (2088825, 84)
RR_sample_90 (553, 84) (104855, 84)
PR_sample_90 (10186, 84) (1518151, 84)
SP_sample_90 (29406, 84) (5581227, 84)
RO_sample_90 (1190, 84) (268544, 84)
BA_sample_90 (13221, 84) (2040175, 84)
SC_sample_90 (3938, 84) (900871, 84)
DF_sample_90 (2044, 84) (402856, 84)
AC_sample_90 (378, 84) (153493, 84)
RS_sample_90 (8369, 84) (1396994, 84)
TO_sample_90 (1132, 84) (247274, 84)
RN_sample_90 (1610, 84) (459515, 84)
AP_sample_90 (1109, 84) (139987, 84)
GO_sample_90 (2485, 84) (829747, 84)
MS_sample_90 (3267, 84) (428235, 84)
PI_sample_90 (3213, 84) (513268, 84)
AM_sample_90 (4705, 84) (702903, 84)
AL_sample_90 (2033, 84) (546991, 84)
SE_sample_90 (3306, 84) (

In [10]:
for key, values in df_90_with_ignored.items():
    df = df_90_with_ignored[key]
    df = df.drop(df[df['n_nu_peso']==9999].index)
    df = df.drop(df[df['n_tp_escolaridade']==0].index)
    df = df.drop(df[df['n_tp_funcao_responsavel']==0].index)
    df = df.drop(columns = {'n_tp_metodo_estimar'}) 
    
    df_90_with_ignored[key] = df
    print(key, df_90_with_ignored[key].shape)

CE_sample_90 (1298835, 83)
MA_sample_90 (1220481, 83)
MT_sample_90 (513328, 83)
MG_sample_90 (2572414, 83)
PE_sample_90 (1468402, 83)
PB_sample_90 (584984, 83)
ES_sample_90 (489529, 83)
PA_sample_90 (1429054, 83)
RJ_sample_90 (2099316, 83)
RR_sample_90 (105406, 83)
PR_sample_90 (1528332, 83)
SP_sample_90 (5610576, 83)
RO_sample_90 (269716, 83)
BA_sample_90 (2053178, 83)
SC_sample_90 (904803, 83)
DF_sample_90 (404879, 83)
AC_sample_90 (153861, 83)
RS_sample_90 (1405346, 83)
TO_sample_90 (248402, 83)
RN_sample_90 (461091, 83)
AP_sample_90 (141092, 83)
GO_sample_90 (832179, 83)
MS_sample_90 (431499, 83)
PI_sample_90 (516436, 83)
AM_sample_90 (707588, 83)
AL_sample_90 (549005, 83)
SE_sample_90 (362627, 83)


In [11]:
for key, values in df_90_with_ignored.items():
    df = df_90_with_ignored[key].copy()
    print(key, df[df['morte_menor_28d']==1].shape, df[df['morte_menor_28d']==0].shape)

CE_sample_90 (5971, 83) (1292864, 83)
MA_sample_90 (6153, 83) (1214328, 83)
MT_sample_90 (3140, 83) (510188, 83)
MG_sample_90 (11119, 83) (2561295, 83)
PE_sample_90 (9434, 83) (1458968, 83)
PB_sample_90 (2121, 83) (582863, 83)
ES_sample_90 (1558, 83) (487971, 83)
PA_sample_90 (9280, 83) (1419774, 83)
RJ_sample_90 (10556, 83) (2088760, 83)
RR_sample_90 (553, 83) (104853, 83)
PR_sample_90 (10186, 83) (1518146, 83)
SP_sample_90 (29405, 83) (5581171, 83)
RO_sample_90 (1190, 83) (268526, 83)
BA_sample_90 (13220, 83) (2039958, 83)
SC_sample_90 (3938, 83) (900865, 83)
DF_sample_90 (2044, 83) (402835, 83)
AC_sample_90 (378, 83) (153483, 83)
RS_sample_90 (8369, 83) (1396977, 83)
TO_sample_90 (1132, 83) (247270, 83)
RN_sample_90 (1610, 83) (459481, 83)
AP_sample_90 (1109, 83) (139983, 83)
GO_sample_90 (2485, 83) (829694, 83)
MS_sample_90 (3267, 83) (428232, 83)
PI_sample_90 (3212, 83) (513224, 83)
AM_sample_90 (4704, 83) (702884, 83)
AL_sample_90 (2033, 83) (546972, 83)
SE_sample_90 (3306, 83) (

### Check point 

<font size="3">Save a version without ignored but with missing</font>

In [ ]:
for key,values in df_90_with_ignored.items():
        df_90_with_ignored[key].to_csv(key+"_sample_90_without_ignored.csv", index=False)
        print(key)

<a id='missing'></a>
## Missing values

<font size="3">Initially let's pick only the columns which contain the suffix "n_", so we are dealing only with records that should contain data from both datasets (SIM and SINASC)</font>

In [12]:
df_sinasc_without_na = {}
for key, values in df_90_with_ignored.items():
    commons = [c for c in df_90_with_ignored[key].columns if c.startswith("n_")]
    perc_common = df_90_with_ignored[key][commons].isnull().mean().sort_index() * 100
    commons.extend(
    [
        "id",
        "ID",
        "UF_id",
        "morte_menor_28d",
        "date_death",
        "day_birth",
        "month_birth",
        "year_birth",
        "day_death",
        "month_death",
        "year_death"
    ])
    df_sinasc_without_na[key] = df_90_with_ignored[key][commons].copy()
    commons.insert(0, "id")
    print(key, perc_common)

CE_sample_90 n_co_bairro_ocorrencia            96.293679
n_co_bairro_residencia            81.059642
n_co_cid                          99.296754
n_co_municipio_ibge_ocorrencia     0.000000
n_co_municipio_ibge_residencia     0.000000
n_dt_nascimento                    0.000000
n_dt_nascimento_mae               55.655953
n_nu_apgar1                        1.543999
n_nu_apgar5                        2.458203
n_nu_consulta_prenatal            64.305551
n_nu_idade                         0.015475
n_nu_kotelchuck                   77.620560
n_nu_mes_gestacao_prenatal        58.049098
n_nu_peso                          0.048120
n_nu_semana_gestacao              57.975493
n_nu_uf_inform                    54.968029
n_paridade                        77.620560
n_qt_gestacao_anterior            60.101091
n_qt_nascidos_mortos              19.146851
n_qt_nascidos_vivos               11.372653
n_qt_parto_cesaria                62.905758
n_qt_parto_normal                 61.913869
n_sg_sexo          

PB_sample_90 n_co_bairro_ocorrencia            97.958577
n_co_bairro_residencia            82.376783
n_co_cid                          99.167499
n_co_municipio_ibge_ocorrencia     0.000000
n_co_municipio_ibge_residencia     0.000000
n_dt_nascimento                    0.000000
n_dt_nascimento_mae               54.964238
n_nu_apgar1                        1.287557
n_nu_apgar5                        2.450836
n_nu_consulta_prenatal            62.962235
n_nu_idade                         0.017094
n_nu_kotelchuck                   78.020595
n_nu_mes_gestacao_prenatal        56.822238
n_nu_peso                          0.050087
n_nu_semana_gestacao              55.870417
n_nu_uf_inform                    54.311742
n_paridade                        78.020595
n_qt_gestacao_anterior            55.791953
n_qt_nascidos_mortos               7.514394
n_qt_nascidos_vivos                4.597733
n_qt_parto_cesaria                56.533854
n_qt_parto_normal                 56.192135
n_sg_sexo          

PR_sample_90 n_co_bairro_ocorrencia            91.489349
n_co_bairro_residencia            68.594127
n_co_cid                          99.306499
n_co_municipio_ibge_ocorrencia     0.000000
n_co_municipio_ibge_residencia     0.000000
n_dt_nascimento                    0.000000
n_dt_nascimento_mae               62.125899
n_nu_apgar1                        0.226587
n_nu_apgar5                        0.210883
n_nu_consulta_prenatal            69.604052
n_nu_idade                         0.002028
n_nu_kotelchuck                   77.196316
n_nu_mes_gestacao_prenatal        62.299160
n_nu_peso                          0.015245
n_nu_semana_gestacao              62.539422
n_nu_uf_inform                    55.722186
n_paridade                        77.196316
n_qt_gestacao_anterior            62.219073
n_qt_nascidos_mortos               0.708616
n_qt_nascidos_vivos                0.492432
n_qt_parto_cesaria                62.260491
n_qt_parto_normal                 62.249302
n_sg_sexo          

DF_sample_90 n_co_bairro_ocorrencia            90.387745
n_co_bairro_residencia            53.659735
n_co_cid                          99.352152
n_co_municipio_ibge_ocorrencia     0.000000
n_co_municipio_ibge_residencia     0.000000
n_dt_nascimento                    0.000000
n_dt_nascimento_mae               56.328434
n_nu_apgar1                        1.788930
n_nu_apgar5                        1.714586
n_nu_consulta_prenatal            65.365702
n_nu_idade                         0.002470
n_nu_kotelchuck                   80.137029
n_nu_mes_gestacao_prenatal        58.406339
n_nu_peso                          0.009386
n_nu_semana_gestacao              57.066432
n_nu_uf_inform                    52.780954
n_paridade                        80.137029
n_qt_gestacao_anterior            58.337923
n_qt_nascidos_mortos              11.464166
n_qt_nascidos_vivos                7.906313
n_qt_parto_cesaria                59.628679
n_qt_parto_normal                 59.044801
n_sg_sexo          

GO_sample_90 n_co_bairro_ocorrencia            95.949069
n_co_bairro_residencia            87.843120
n_co_cid                          99.374293
n_co_municipio_ibge_ocorrencia     0.000000
n_co_municipio_ibge_residencia     0.000000
n_dt_nascimento                    0.000000
n_dt_nascimento_mae               56.409138
n_nu_apgar1                        0.987408
n_nu_apgar5                        0.960851
n_nu_consulta_prenatal            65.384731
n_nu_idade                         0.009974
n_nu_kotelchuck                   77.788433
n_nu_mes_gestacao_prenatal        58.144822
n_nu_peso                          0.016102
n_nu_semana_gestacao              58.090026
n_nu_uf_inform                    55.614357
n_paridade                        77.788433
n_qt_gestacao_anterior            60.254224
n_qt_nascidos_mortos              11.633795
n_qt_nascidos_vivos                7.241231
n_qt_parto_cesaria                62.177248
n_qt_parto_normal                 61.970562
n_sg_sexo          

<font size="3">Categorical Variable: Input missing data with mode</font> 

In [13]:
exclude_mode_cols = [
    "n_nu_idade",
    "n_qt_nascidos_vivos",
    "n_qt_nascidos_mortos",
    "n_dt_nascimento",
    "n_nu_apgar1",
    "n_nu_apgar5",
    "n_nu_peso",
    "n_qt_gestacao_anterior",
    "n_qt_parto_normal",
    "n_qt_parto_cesarea",
    "n_nu_semana_gestacao",
    "date_death",
    "day_death",
    "month_death",
    "year_death"
    ]

In [14]:
for key, values in df_sinasc_without_na.items():
   
    dif_col_mode = [c for c in df_sinasc_without_na[key].columns if c not in exclude_mode_cols]

In [15]:
for key, values in df_sinasc_without_na.items():
    for column in dif_col_mode:
        
        df_sinasc_without_na[key][column].fillna(
            df_sinasc_without_na[key][column].mode()[0], inplace=True)
        
    print(key, df_sinasc_without_na[key].shape)

CE_sample_90 (1298835, 52)
MA_sample_90 (1220481, 52)
MT_sample_90 (513328, 52)
MG_sample_90 (2572414, 52)
PE_sample_90 (1468402, 52)
PB_sample_90 (584984, 52)
ES_sample_90 (489529, 52)
PA_sample_90 (1429054, 52)
RJ_sample_90 (2099316, 52)
RR_sample_90 (105406, 52)
PR_sample_90 (1528332, 52)
SP_sample_90 (5610576, 52)
RO_sample_90 (269716, 52)
BA_sample_90 (2053178, 52)
SC_sample_90 (904803, 52)
DF_sample_90 (404879, 52)
AC_sample_90 (153861, 52)
RS_sample_90 (1405346, 52)
TO_sample_90 (248402, 52)
RN_sample_90 (461091, 52)
AP_sample_90 (141092, 52)
GO_sample_90 (832179, 52)
MS_sample_90 (431499, 52)
PI_sample_90 (516436, 52)
AM_sample_90 (707588, 52)
AL_sample_90 (549005, 52)
SE_sample_90 (362627, 52)


<font size="3">Numerical Variable: input missing data with mean</font>

In [16]:
perform_mean_cols = [
        "n_nu_idade",
        "n_qt_nascidos_vivos",
        "n_qt_nascidos_mortos",
        "n_nu_apgar1",
        "n_nu_apgar5",
        "n_nu_peso",
        "n_qt_gestacao_anterior",
        "n_qt_parto_normal",
        "n_qt_parto_cesaria",
        "n_nu_semana_gestacao"
         ]

In [17]:
for key, values in df_sinasc_without_na.items():

    for column in perform_mean_cols:
        df_sinasc_without_na[key][column].fillna(int(df_sinasc_without_na[key][column].mean()), inplace=True)
    print(key,df_sinasc_without_na[key].shape)

CE_sample_90 (1298835, 52)
MA_sample_90 (1220481, 52)
MT_sample_90 (513328, 52)
MG_sample_90 (2572414, 52)
PE_sample_90 (1468402, 52)
PB_sample_90 (584984, 52)
ES_sample_90 (489529, 52)
PA_sample_90 (1429054, 52)
RJ_sample_90 (2099316, 52)
RR_sample_90 (105406, 52)
PR_sample_90 (1528332, 52)
SP_sample_90 (5610576, 52)
RO_sample_90 (269716, 52)
BA_sample_90 (2053178, 52)
SC_sample_90 (904803, 52)
DF_sample_90 (404879, 52)
AC_sample_90 (153861, 52)
RS_sample_90 (1405346, 52)
TO_sample_90 (248402, 52)
RN_sample_90 (461091, 52)
AP_sample_90 (141092, 52)
GO_sample_90 (832179, 52)
MS_sample_90 (431499, 52)
PI_sample_90 (516436, 52)
AM_sample_90 (707588, 52)
AL_sample_90 (549005, 52)
SE_sample_90 (362627, 52)


<font size="3">Check if all variables starting with n_ are without missing now</font>

In [18]:
for key, values in df_sinasc_without_na.items():
    print(key, df_sinasc_without_na[key].isnull().mean() * 100)

CE_sample_90 n_tp_ocorrencia                    0.00000
n_co_municipio_ibge_ocorrencia     0.00000
n_nu_idade                         0.00000
n_tp_estado_civil                  0.00000
n_tp_escolaridade                  0.00000
n_qt_nascidos_vivos                0.00000
n_qt_nascidos_mortos               0.00000
n_co_municipio_ibge_residencia     0.00000
n_tp_gestacao                      0.00000
n_tp_gravidez                      0.00000
n_tp_parto                         0.00000
n_tp_prenatal                      0.00000
n_dt_nascimento                    0.00000
n_sg_sexo                          0.00000
n_nu_apgar1                        0.00000
n_nu_apgar5                        0.00000
n_tp_raca_cor                      0.00000
n_nu_peso                          0.00000
n_co_cid                           0.00000
n_st_malformacao                   0.00000
n_co_bairro_ocorrencia             0.00000
n_co_bairro_residencia             0.00000
n_nu_uf_inform                     0.0000

PE_sample_90 n_tp_ocorrencia                    0.000000
n_co_municipio_ibge_ocorrencia     0.000000
n_nu_idade                         0.000000
n_tp_estado_civil                  0.000000
n_tp_escolaridade                  0.000000
n_qt_nascidos_vivos                0.000000
n_qt_nascidos_mortos               0.000000
n_co_municipio_ibge_residencia     0.000000
n_tp_gestacao                      0.000000
n_tp_gravidez                      0.000000
n_tp_parto                         0.000000
n_tp_prenatal                      0.000000
n_dt_nascimento                    0.000000
n_sg_sexo                          0.000000
n_nu_apgar1                        0.000000
n_nu_apgar5                        0.000000
n_tp_raca_cor                      0.000000
n_nu_peso                          0.000000
n_co_cid                           0.000000
n_st_malformacao                   0.000000
n_co_bairro_ocorrencia             0.000000
n_co_bairro_residencia             0.000000
n_nu_uf_inform     

RJ_sample_90 n_tp_ocorrencia                    0.00000
n_co_municipio_ibge_ocorrencia     0.00000
n_nu_idade                         0.00000
n_tp_estado_civil                  0.00000
n_tp_escolaridade                  0.00000
n_qt_nascidos_vivos                0.00000
n_qt_nascidos_mortos               0.00000
n_co_municipio_ibge_residencia     0.00000
n_tp_gestacao                      0.00000
n_tp_gravidez                      0.00000
n_tp_parto                         0.00000
n_tp_prenatal                      0.00000
n_dt_nascimento                    0.00000
n_sg_sexo                          0.00000
n_nu_apgar1                        0.00000
n_nu_apgar5                        0.00000
n_tp_raca_cor                      0.00000
n_nu_peso                          0.00000
n_co_cid                           0.00000
n_co_bairro_ocorrencia             0.00000
n_co_bairro_residencia             0.00000
n_st_malformacao                   0.00000
n_nu_uf_inform                     0.0000

BA_sample_90 n_tp_ocorrencia                    0.00000
n_co_bairro_ocorrencia             0.00000
n_co_municipio_ibge_ocorrencia     0.00000
n_nu_idade                         0.00000
n_tp_estado_civil                  0.00000
n_tp_escolaridade                  0.00000
n_qt_nascidos_vivos                0.00000
n_qt_nascidos_mortos               0.00000
n_co_bairro_residencia             0.00000
n_co_municipio_ibge_residencia     0.00000
n_tp_gestacao                      0.00000
n_tp_gravidez                      0.00000
n_tp_parto                         0.00000
n_tp_prenatal                      0.00000
n_dt_nascimento                    0.00000
n_sg_sexo                          0.00000
n_nu_apgar1                        0.00000
n_nu_apgar5                        0.00000
n_tp_raca_cor                      0.00000
n_nu_peso                          0.00000
n_st_malformacao                   0.00000
n_co_cid                           0.00000
n_nu_uf_inform                     0.0000

RS_sample_90 n_tp_ocorrencia                    0.000000
n_co_municipio_ibge_ocorrencia     0.000000
n_nu_idade                         0.000000
n_tp_estado_civil                  0.000000
n_tp_escolaridade                  0.000000
n_qt_nascidos_vivos                0.000000
n_qt_nascidos_mortos               0.000000
n_co_municipio_ibge_residencia     0.000000
n_tp_gestacao                      0.000000
n_tp_gravidez                      0.000000
n_tp_parto                         0.000000
n_tp_prenatal                      0.000000
n_dt_nascimento                    0.000000
n_sg_sexo                          0.000000
n_nu_apgar1                        0.000000
n_nu_apgar5                        0.000000
n_tp_raca_cor                      0.000000
n_nu_peso                          0.000000
n_co_cid                           0.000000
n_st_malformacao                   0.000000
n_co_bairro_ocorrencia             0.000000
n_co_bairro_residencia             0.000000
n_nu_uf_inform     

GO_sample_90 n_tp_ocorrencia                    0.000000
n_co_municipio_ibge_ocorrencia     0.000000
n_nu_idade                         0.000000
n_tp_estado_civil                  0.000000
n_tp_escolaridade                  0.000000
n_qt_nascidos_vivos                0.000000
n_qt_nascidos_mortos               0.000000
n_co_municipio_ibge_residencia     0.000000
n_tp_gestacao                      0.000000
n_tp_gravidez                      0.000000
n_tp_parto                         0.000000
n_tp_prenatal                      0.000000
n_dt_nascimento                    0.000000
n_sg_sexo                          0.000000
n_nu_apgar1                        0.000000
n_nu_apgar5                        0.000000
n_tp_raca_cor                      0.000000
n_nu_peso                          0.000000
n_co_cid                           0.000000
n_st_malformacao                   0.000000
n_co_bairro_ocorrencia             0.000000
n_co_bairro_residencia             0.000000
n_nu_uf_inform     

SE_sample_90 n_tp_ocorrencia                    0.000000
n_co_municipio_ibge_ocorrencia     0.000000
n_nu_idade                         0.000000
n_tp_estado_civil                  0.000000
n_tp_escolaridade                  0.000000
n_qt_nascidos_vivos                0.000000
n_qt_nascidos_mortos               0.000000
n_co_municipio_ibge_residencia     0.000000
n_tp_gestacao                      0.000000
n_tp_gravidez                      0.000000
n_tp_parto                         0.000000
n_tp_prenatal                      0.000000
n_dt_nascimento                    0.000000
n_sg_sexo                          0.000000
n_nu_apgar1                        0.000000
n_nu_apgar5                        0.000000
n_tp_raca_cor                      0.000000
n_nu_peso                          0.000000
n_co_cid                           0.000000
n_st_malformacao                   0.000000
n_co_bairro_ocorrencia             0.000000
n_co_bairro_residencia             0.000000
n_nu_uf_inform     

- Cast to numeric;
- Cast to Int64;

In [19]:
for key, values in df_sinasc_without_na.items():
    for column in df_sinasc_without_na[key].columns:
        if "_qt" in column or "_nu" in column or "_tp" in column or "_st" in column:
                df_sinasc_without_na[key][column] = pd.to_numeric(
                df_sinasc_without_na[key][column], errors="coerce")

In [20]:
for key, values in df_sinasc_without_na.items():
    for column in df_sinasc_without_na[key].columns:
        if "_qt" in column or "_nu" in column or "_tp" in column or "_st" in column:            
            df_sinasc_without_na[key][column] = df_sinasc_without_na[key][column].astype("Int64")

In [28]:
for key, values in df_sinasc_without_na.items():
    print(key, df_sinasc_without_na[key].dtypes)

CE_sample_90 n_tp_ocorrencia                     Int64
n_co_municipio_ibge_ocorrencia      int64
n_nu_idade                          Int64
n_tp_estado_civil                   Int64
n_tp_escolaridade                   Int64
n_qt_nascidos_vivos                 Int64
n_qt_nascidos_mortos                Int64
n_co_municipio_ibge_residencia      int64
n_tp_gestacao                       Int64
n_tp_gravidez                       Int64
n_tp_parto                          Int64
n_tp_prenatal                       Int64
n_dt_nascimento                     int64
n_sg_sexo                          object
n_nu_apgar1                         Int64
n_nu_apgar5                         Int64
n_tp_raca_cor                       Int64
n_nu_peso                           Int64
n_co_cid                           object
n_st_malformacao                    Int64
n_co_bairro_ocorrencia            float64
n_co_bairro_residencia            float64
n_nu_uf_inform                      Int64
n_dt_nascimento_mae  

In [ ]:
for key,values in df_sinasc_without_na.items():
    df_sinasc_without_na[key].to_csv(key+"_without_na_and_ignored.csv")
    print(key)